# Candidate Low-Altitude Point Events — `point_event_v1`

This notebook constructs the frozen Phase 1 physical modeling dataset from the full-year five-minute Flightradar24 JSONL collection.

**Primary modeling unit:** one deduplicated candidate low-altitude aircraft observation.

A retained row must:

- be within **1.0 statute mile** of the PANC reference point;
- have altitude **greater than or equal to 100 feet and less than 1,000 feet**;
- have ground speed **greater than or equal to 30 knots**;
- contain valid UTC snapshot and aircraft-event timestamps;
- contain valid geographic coordinates; and
- contain at least one usable aircraft or track identifier.

These rules identify **candidate events**, not confirmed anomalies or threats. Episode aggregation, recurrence features, cyber alignment, labels, and modeling belong in later notebooks.

## 1. Imports and project paths

The notebook locates the project root by finding `data/fr24_raw_5min`. It is intended to sit at the top level of the existing **Jupyter Notebook** project, beside the `config` directory.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
from typing import Any

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start: Path) -> Path:
    """Find the nearest directory containing the expected raw-data folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "fr24_raw_5min").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/fr24_raw_5min. Run this notebook from the project "
        "directory or one of its subdirectories."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "config" / "point_event_v1.yaml"

print(f"Project root: {PROJECT_ROOT}")
print(f"Configuration: {CONFIG_PATH}")

Project root: C:\Users\slipk\OneDrive - The George Washington University\DC8\Praxis\Project\Jupyter Notebook
Configuration: C:\Users\slipk\OneDrive - The George Washington University\DC8\Praxis\Project\Jupyter Notebook\config\point_event_v1.yaml


## 2. Load and validate the frozen configuration

All thresholds, paths, identity precedence, deduplication behavior, and regression expectations are read from `config/point_event_v1.yaml`. Changing a substantive selection rule requires a new definition version rather than silently changing `point_event_v1`.

In [2]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing frozen configuration: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

assert config["definition"]["version"] == "point_event_v1"
assert config["definition"]["status"] == "frozen"
assert config["labels"]["anomaly_label_applied_during_selection"] is False
assert config["secondary_context"]["collapse_events_into_episode_rows"] is False

RAW_DIR = PROJECT_ROOT / config["paths"]["raw_data_dir"]
OUTPUT_DIR = PROJECT_ROOT / config["paths"]["output_dir"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "definition": config["definition"],
    "study_site": config["study_site"],
    "filters": config["filters"],
    "time": config["time"],
    "raw_data_dir": str(RAW_DIR),
    "output_dir": str(OUTPUT_DIR),
}, indent=2))

{
  "definition": {
    "name": "candidate_low_altitude_point_event",
    "version": "point_event_v1",
    "status": "frozen",
    "primary_modeling_unit": "one_qualifying_aircraft_observation",
    "description": "One deduplicated aircraft observation within one statute mile of PANC, at 100 through 999 feet, with ground speed of at least 30 knots, valid UTC timestamps and coordinates, and a usable aircraft or track identifier."
  },
  "study_site": {
    "site_code": "PANC",
    "latitude": 61.1743,
    "longitude": -149.9982,
    "distance_method": "haversine",
    "radius_miles_inclusive": 1.0
  },
  "filters": {
    "altitude_min_ft_inclusive": 100,
    "altitude_max_ft_exclusive": 1000,
    "ground_speed_min_kt_inclusive": 30
  },
  "time": {
    "source_snapshot_interval_s": 300,
    "primary_event_time_field": "aircraft_event_time_utc",
    "audit_time_field": "snapshot_time_utc",
    "timezone": "UTC",
    "stale_event_threshold_s": 30
  },
  "raw_data_dir": "C:\\Users\\slipk\\

## 3. Helper functions

The helper functions normalize identifiers, calculate Haversine distance, generate deterministic event identifiers, and create reproducibility checksums.

In [3]:
def normalize_identifier(value: Any) -> str | None:
    """Return an uppercase, stripped identifier or None."""
    if value is None or pd.isna(value):
        return None
    normalized = str(value).strip().upper()
    return normalized or None


def haversine_miles_vectorized(
    lat1: float,
    lon1: float,
    lat2: pd.Series,
    lon2: pd.Series,
) -> pd.Series:
    """Calculate great-circle distance in statute miles."""
    earth_radius_miles = 3958.8
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2.astype(float))
    delta_phi = np.radians(lat2.astype(float) - lat1)
    delta_lambda = np.radians(lon2.astype(float) - lon1)
    a = (
        np.sin(delta_phi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2
    )
    return pd.Series(
        2.0 * earth_radius_miles * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a)),
        index=lat2.index,
    )


def timestamp_token(value: pd.Timestamp) -> str:
    """Create a stable ISO-8601 timestamp token in UTC."""
    if pd.isna(value):
        raise ValueError("Cannot create an event identifier from a missing timestamp.")
    return value.tz_convert("UTC").isoformat()


def make_event_id(row: pd.Series) -> str:
    """Create a deterministic point-event identifier from stable attributes."""
    canonical = "|".join(
        [
            config["definition"]["version"],
            str(row["aircraft_key"]),
            timestamp_token(row["snapshot_time_utc"]),
            timestamp_token(row["aircraft_event_time_utc"]),
            f'{float(row["latitude"]):.6f}',
            f'{float(row["longitude"]):.6f}',
        ]
    )
    digest = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return f"PEV1-{digest[:16].upper()}"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def source_manifest_sha256(files: list[Path]) -> str:
    """Hash sorted filename/file-hash pairs to fingerprint the raw directory."""
    manifest = hashlib.sha256()
    for path in sorted(files, key=lambda p: p.name):
        manifest.update(path.name.encode("utf-8"))
        manifest.update(b"|")
        manifest.update(sha256_file(path).encode("ascii"))
        manifest.update(b"\n")
    return manifest.hexdigest()

## 4. Inventory and flatten the raw JSONL collection

This section performs a single pass over all daily files. It preserves snapshot-level audit information and creates one intermediate row for every returned aircraft observation. Empty snapshots remain in the collection audit but do not become aircraft-event rows.

In [4]:
jsonl_files = sorted(RAW_DIR.glob("*.jsonl"))
if not jsonl_files:
    raise FileNotFoundError(f"No JSONL files found in {RAW_DIR}")

snapshot_rows: list[dict[str, Any]] = []
aircraft_rows: list[dict[str, Any]] = []
malformed_rows: list[dict[str, Any]] = []

for source_file in jsonl_files:
    with source_file.open("r", encoding="utf-8") as handle:
        for source_line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                malformed_rows.append(
                    {
                        "source_filename": source_file.name,
                        "source_line_number": source_line_number,
                        "error": str(exc),
                        "raw_excerpt": line[:500],
                    }
                )
                continue

            data = record.get("data")
            if not isinstance(data, list):
                data = []

            snapshot_rows.append(
                {
                    "source_filename": source_file.name,
                    "source_line_number": source_line_number,
                    "snapshot_timestamp_raw": record.get("timestamp"),
                    "bounds": record.get("bounds"),
                    "altitude_ranges": json.dumps(record.get("altitude_ranges")),
                    "aircraft_count": len(data),
                }
            )

            for source_aircraft_index, aircraft in enumerate(data):
                aircraft_rows.append(
                    {
                        "source_filename": source_file.name,
                        "source_line_number": source_line_number,
                        "source_aircraft_index": source_aircraft_index,
                        "snapshot_timestamp_raw": record.get("timestamp"),
                        "fr24_id_raw": aircraft.get("fr24_id"),
                        "hex_raw": aircraft.get("hex"),
                        "callsign_raw": aircraft.get("callsign"),
                        "aircraft_timestamp_raw": aircraft.get("timestamp"),
                        "latitude_raw": aircraft.get("lat"),
                        "longitude_raw": aircraft.get("lon"),
                        "altitude_raw": aircraft.get("alt"),
                        "ground_speed_raw": aircraft.get("gspeed"),
                        "vertical_speed_raw": aircraft.get("vspeed"),
                        "track_raw": aircraft.get("track"),
                        "squawk_raw": aircraft.get("squawk"),
                        "source_type_raw": aircraft.get("source"),
                    }
                )

snapshot_df = pd.DataFrame(snapshot_rows)
raw_df = pd.DataFrame(aircraft_rows)
malformed_df = pd.DataFrame(malformed_rows)

snapshot_df["snapshot_time_utc"] = pd.to_datetime(
    snapshot_df["snapshot_timestamp_raw"], errors="coerce", utc=True
)

inventory_summary = pd.DataFrame(
    {
        "metric": [
            "Daily JSONL files",
            "Snapshot records",
            "Raw aircraft observations",
            "Empty snapshots",
            "Malformed JSON rows",
            "Maximum aircraft in one snapshot",
        ],
        "value": [
            len(jsonl_files),
            len(snapshot_df),
            len(raw_df),
            int((snapshot_df["aircraft_count"] == 0).sum()),
            len(malformed_df),
            int(snapshot_df["aircraft_count"].max()),
        ],
    }
)
inventory_summary

,metric,value
0,Daily JSONL files,365
1,Snapshot records,105120
2,Raw aircraft observations,30107
3,Empty snapshots,78949
4,Malformed JSON rows,0
5,Maximum aircraft in one snapshot,5


## 5. Validate full-year collection integrity

The frozen 2025 collection is expected to contain one file per UTC date, 288 five-minute snapshots per file, and 105,120 unique snapshot timestamps.

In [5]:
source_year = int(config["regression_expectations"]["source_year"])
interval_seconds = int(config["time"]["source_snapshot_interval_s"])

file_snapshot_counts = snapshot_df.groupby("source_filename").size()
invalid_file_counts = file_snapshot_counts[file_snapshot_counts != 86400 // interval_seconds]

duplicate_snapshot_count = int(snapshot_df["snapshot_time_utc"].duplicated().sum())
invalid_snapshot_timestamp_count = int(snapshot_df["snapshot_time_utc"].isna().sum())

expected_timestamps = pd.date_range(
    start=f"{source_year}-01-01 00:00:00+00:00",
    end=f"{source_year}-12-31 23:55:00+00:00",
    freq=f"{interval_seconds}s",
)
actual_timestamps = pd.DatetimeIndex(snapshot_df["snapshot_time_utc"].dropna().sort_values().unique())
missing_timestamps = expected_timestamps.difference(actual_timestamps)
unexpected_timestamps = actual_timestamps.difference(expected_timestamps)

collection_checks = pd.DataFrame(
    {
        "check": [
            "Files with unexpected snapshot counts",
            "Invalid snapshot timestamps",
            "Duplicate snapshot timestamps",
            "Missing expected timestamps",
            "Unexpected timestamps outside 2025 grid",
            "Malformed JSON rows",
        ],
        "count": [
            len(invalid_file_counts),
            invalid_snapshot_timestamp_count,
            duplicate_snapshot_count,
            len(missing_timestamps),
            len(unexpected_timestamps),
            len(malformed_df),
        ],
    }
)
collection_checks

,check,count
0,Files with unexpected snapshot counts,0
1,Invalid snapshot timestamps,0
2,Duplicate snapshot timestamps,0
3,Missing expected timestamps,0
4,Unexpected timestamps outside 2025 grid,0
5,Malformed JSON rows,0


In [6]:
assert len(invalid_file_counts) == 0, invalid_file_counts.to_dict()
assert invalid_snapshot_timestamp_count == 0
assert duplicate_snapshot_count == 0
assert len(missing_timestamps) == 0, missing_timestamps[:20].tolist()
assert len(unexpected_timestamps) == 0, unexpected_timestamps[:20].tolist()
assert len(malformed_df) == 0

print("Collection-integrity checks passed.")

Collection-integrity checks passed.


## 6. Normalize timestamps, identifiers, measurements, and distance

`flight_track_id` preserves the source `fr24_id`. `aircraft_key` uses the frozen stable-identity precedence: hex code, otherwise callsign, otherwise `fr24_id`.

In [7]:
working = raw_df.copy()

working["snapshot_time_utc"] = pd.to_datetime(
    working["snapshot_timestamp_raw"], errors="coerce", utc=True
)
working["aircraft_event_time_utc"] = pd.to_datetime(
    working["aircraft_timestamp_raw"], errors="coerce", utc=True
)
working["event_lag_seconds"] = (
    working["aircraft_event_time_utc"] - working["snapshot_time_utc"]
).dt.total_seconds()
working["absolute_event_lag_seconds"] = working["event_lag_seconds"].abs()

working["fr24_id"] = working["fr24_id_raw"].map(normalize_identifier)
working["hex"] = working["hex_raw"].map(normalize_identifier)
working["callsign"] = working["callsign_raw"].map(normalize_identifier)
working["flight_track_id"] = working["fr24_id"]

working["aircraft_key"] = working["hex"].fillna(working["callsign"]).fillna(working["fr24_id"])
working["aircraft_key_source"] = np.select(
    [working["hex"].notna(), working["callsign"].notna(), working["fr24_id"].notna()],
    ["hex", "callsign", "fr24_id"],
    default=None,
)

working["latitude"] = pd.to_numeric(working["latitude_raw"], errors="coerce")
working["longitude"] = pd.to_numeric(working["longitude_raw"], errors="coerce")
working["altitude_ft"] = pd.to_numeric(working["altitude_raw"], errors="coerce")
working["ground_speed_kt"] = pd.to_numeric(working["ground_speed_raw"], errors="coerce")
working["vertical_speed_fpm"] = pd.to_numeric(working["vertical_speed_raw"], errors="coerce")
working["track_degrees"] = pd.to_numeric(working["track_raw"], errors="coerce")
working["squawk"] = working["squawk_raw"].map(lambda x: None if x is None else str(x).strip())
working["source_type"] = working["source_type_raw"].map(normalize_identifier)

valid_coordinate_mask = (
    working["latitude"].between(-90, 90, inclusive="both")
    & working["longitude"].between(-180, 180, inclusive="both")
)
working["distance_miles"] = np.nan
working.loc[valid_coordinate_mask, "distance_miles"] = haversine_miles_vectorized(
    float(config["study_site"]["latitude"]),
    float(config["study_site"]["longitude"]),
    working.loc[valid_coordinate_mask, "latitude"],
    working.loc[valid_coordinate_mask, "longitude"],
)

working["stale_event_flag"] = (
    working["absolute_event_lag_seconds"] > float(config["time"]["stale_event_threshold_s"])
).astype("int8")
working["missing_hex_flag"] = working["hex"].isna().astype("int8")
working["missing_callsign_flag"] = working["callsign"].isna().astype("int8")

working[[
    "snapshot_time_utc", "aircraft_event_time_utc", "event_lag_seconds",
    "aircraft_key", "aircraft_key_source", "latitude", "longitude",
    "distance_miles", "altitude_ft", "ground_speed_kt", "source_type"
]].head()

## 7. Apply the frozen selection rules in a deterministic order

Each excluded aircraft observation receives one primary rejection reason. No anomaly label or anomaly score participates in event selection.

In [8]:
radius = float(config["study_site"]["radius_miles_inclusive"])
altitude_min = float(config["filters"]["altitude_min_ft_inclusive"])
altitude_max = float(config["filters"]["altitude_max_ft_exclusive"])
ground_speed_min = float(config["filters"]["ground_speed_min_kt_inclusive"])

working["rejection_reason"] = pd.Series(pd.NA, index=working.index, dtype="string")


def reject(mask: pd.Series, reason: str) -> None:
    eligible = working["rejection_reason"].isna() & mask.fillna(False)
    working.loc[eligible, "rejection_reason"] = reason


reject(working["snapshot_time_utc"].isna(), "invalid_snapshot_timestamp")
reject(working["aircraft_event_time_utc"].isna(), "invalid_aircraft_event_timestamp")
reject(~valid_coordinate_mask, "invalid_coordinates")
reject(working["aircraft_key"].isna(), "missing_identifier")
reject(working["distance_miles"].isna() | (working["distance_miles"] > radius), "outside_radius")
reject(working["altitude_ft"].isna(), "invalid_altitude")
reject(working["altitude_ft"] < altitude_min, "altitude_below_minimum")
reject(working["altitude_ft"] >= altitude_max, "altitude_at_or_above_maximum")
reject(working["ground_speed_kt"].isna(), "invalid_ground_speed")
reject(working["ground_speed_kt"] < ground_speed_min, "ground_speed_below_minimum")

pre_dedup = working[working["rejection_reason"].isna()].copy()
nonqualifying_rejections = working[working["rejection_reason"].notna()].copy()

pre_dedup["event_definition_version"] = config["definition"]["version"]
pre_dedup["candidate_event_flag"] = int(config["labels"]["candidate_event_flag"])
pre_dedup["event_id"] = pre_dedup.apply(make_event_id, axis=1)

selection_summary = pd.concat(
    [
        nonqualifying_rejections["rejection_reason"].value_counts().rename("count"),
        pd.Series({"qualifying_before_deduplication": len(pre_dedup)}, name="count"),
    ]
).rename_axis("outcome").reset_index()
selection_summary

,outcome,count
0,outside_radius,15640
1,ground_speed_below_minimum,1604
2,altitude_below_minimum,572
3,altitude_at_or_above_maximum,105
4,qualifying_before_deduplication,12186


## 8. Deduplicate same-aircraft, same-snapshot records

The frozen audit prioritizes direct ADS-B records, then UAT, then MLAT. Within the same source priority, the record closest to the requested snapshot time is retained. Remaining ties are broken by deterministic source-record fields.

In [9]:
source_priority_map = {
    normalize_identifier(key): int(value)
    for key, value in config["deduplication"]["source_priority"].items()
}
default_source_priority = int(config["deduplication"]["default_source_priority"])

pre_dedup["source_priority"] = (
    pre_dedup["source_type"].map(source_priority_map).fillna(default_source_priority).astype(int)
)

sort_columns = [
    "aircraft_key",
    "snapshot_time_utc",
    "source_priority",
    "absolute_event_lag_seconds",
    "aircraft_event_time_utc",
    "latitude",
    "longitude",
    "fr24_id",
    "source_filename",
    "source_line_number",
    "source_aircraft_index",
]

sorted_candidates = pre_dedup.sort_values(sort_columns, kind="mergesort").copy()
group_fields = config["deduplication"]["group_fields"]
sorted_candidates["duplicate_group_size"] = sorted_candidates.groupby(group_fields)["event_id"].transform("size")

keep_mask = ~sorted_candidates.duplicated(subset=group_fields, keep="first")
candidate_events = sorted_candidates[keep_mask].copy()
duplicate_rejections = sorted_candidates[~keep_mask].copy()

candidate_events["deduplication_status"] = np.where(
    candidate_events["duplicate_group_size"] > 1,
    "retained_from_duplicate_group",
    "unique",
)

retained_lookup = candidate_events.set_index(group_fields)["event_id"]
duplicate_rejections["retained_event_id"] = [
    retained_lookup.loc[tuple(row[field] for field in group_fields)]
    for _, row in duplicate_rejections.iterrows()
]
duplicate_rejections["rejection_reason"] = "duplicate_aircraft_snapshot"

print(f"Qualifying before deduplication: {len(pre_dedup):,}")
print(f"Duplicate rows removed: {len(duplicate_rejections):,}")
print(f"Candidate point events retained: {len(candidate_events):,}")

duplicate_rejections[[
    "aircraft_key", "snapshot_time_utc", "event_id", "retained_event_id",
    "source_type", "event_lag_seconds", "rejection_reason"
]]

Qualifying before deduplication: 12,186
Duplicate rows removed: 1
Candidate point events retained: 12,185


## 9. Finalize the authoritative point-event and rejection schemas

The base event table intentionally excludes episode duration, recurrence features, cyber features, and anomaly labels. Those are derived later without replacing the point-event rows.

In [10]:
final_event_columns = [
    "event_id",
    "event_definition_version",
    "candidate_event_flag",
    "aircraft_key",
    "aircraft_key_source",
    "flight_track_id",
    "hex",
    "callsign",
    "fr24_id",
    "snapshot_time_utc",
    "aircraft_event_time_utc",
    "event_lag_seconds",
    "stale_event_flag",
    "latitude",
    "longitude",
    "distance_miles",
    "altitude_ft",
    "ground_speed_kt",
    "vertical_speed_fpm",
    "track_degrees",
    "source_type",
    "squawk",
    "missing_hex_flag",
    "missing_callsign_flag",
    "deduplication_status",
    "source_filename",
    "source_line_number",
    "source_aircraft_index",
]

candidate_events = (
    candidate_events[final_event_columns]
    .sort_values(["aircraft_event_time_utc", "aircraft_key", "event_id"], kind="mergesort")
    .reset_index(drop=True)
)

rejection_columns = [
    "rejection_reason",
    "retained_event_id",
    "aircraft_key",
    "aircraft_key_source",
    "flight_track_id",
    "hex",
    "callsign",
    "fr24_id",
    "snapshot_time_utc",
    "aircraft_event_time_utc",
    "event_lag_seconds",
    "latitude",
    "longitude",
    "distance_miles",
    "altitude_ft",
    "ground_speed_kt",
    "vertical_speed_fpm",
    "track_degrees",
    "source_type",
    "squawk",
    "source_filename",
    "source_line_number",
    "source_aircraft_index",
]

nonqualifying_rejections["retained_event_id"] = pd.NA
all_rejections = pd.concat(
    [nonqualifying_rejections, duplicate_rejections],
    ignore_index=True,
    sort=False,
)
for column in rejection_columns:
    if column not in all_rejections.columns:
        all_rejections[column] = pd.NA
all_rejections = all_rejections[rejection_columns].sort_values(
    ["snapshot_time_utc", "source_filename", "source_line_number", "source_aircraft_index"],
    kind="mergesort",
    na_position="last",
).reset_index(drop=True)

candidate_events.head()

## 10. Quality-control and frozen-regression checks

These assertions prevent a changed rule, incomplete source directory, duplicate key, or unexpected output count from silently replacing the accepted Phase 1 dataset.

In [11]:
assert candidate_events["event_id"].is_unique
assert candidate_events["event_definition_version"].eq("point_event_v1").all()
assert candidate_events["candidate_event_flag"].eq(1).all()
assert candidate_events["aircraft_key"].notna().all()
assert candidate_events["snapshot_time_utc"].notna().all()
assert candidate_events["aircraft_event_time_utc"].notna().all()

assert candidate_events["distance_miles"].le(radius).all()
assert candidate_events["altitude_ft"].ge(altitude_min).all()
assert candidate_events["altitude_ft"].lt(altitude_max).all()
assert candidate_events["ground_speed_kt"].ge(ground_speed_min).all()
assert candidate_events["latitude"].between(-90, 90, inclusive="both").all()
assert candidate_events["longitude"].between(-180, 180, inclusive="both").all()
assert not candidate_events.duplicated(group_fields).any()

expectations = config["regression_expectations"]
actuals = {
    "daily_files": len(jsonl_files),
    "snapshots": len(snapshot_df),
    "raw_aircraft_observations": len(raw_df),
    "qualifying_before_deduplication": len(pre_dedup),
    "qualifying_after_deduplication": len(candidate_events),
}

regression_table = pd.DataFrame(
    [
        {
            "metric": metric,
            "expected": int(expectations[metric]),
            "actual": int(actual),
            "match": int(expectations[metric]) == int(actual),
        }
        for metric, actual in actuals.items()
    ]
)

if bool(expectations.get("strict", True)):
    failures = regression_table[~regression_table["match"]]
    assert failures.empty, failures.to_dict(orient="records")

print("All row-level and frozen-regression checks passed.")
regression_table

All row-level and frozen-regression checks passed.


,metric,expected,actual,match
0,daily_files,365,365,True
1,snapshots,105120,105120,True
2,raw_aircraft_observations,30107,30107,True
3,qualifying_before_deduplication,12186,12186,True
4,qualifying_after_deduplication,12185,12185,True


## 11. Build the data dictionary

In [12]:
data_dictionary_rows = [
    ("event_id", "string", "Deterministic identifier for the retained point event."),
    ("event_definition_version", "string", "Frozen selection-definition version; always point_event_v1."),
    ("candidate_event_flag", "integer", "Selection status; always 1 in the authoritative event table."),
    ("aircraft_key", "string", "Stable aircraft identity using hex, then callsign, then fr24_id."),
    ("aircraft_key_source", "string", "Identifier field used to construct aircraft_key."),
    ("flight_track_id", "string", "Original FR24 track identifier; equal to normalized fr24_id."),
    ("hex", "string", "Normalized ICAO hex identifier when available."),
    ("callsign", "string", "Normalized callsign when available."),
    ("fr24_id", "string", "Normalized Flightradar24 record or track identifier."),
    ("snapshot_time_utc", "datetime UTC", "Requested five-minute collection timestamp."),
    ("aircraft_event_time_utc", "datetime UTC", "Aircraft-reported observation timestamp used as event time."),
    ("event_lag_seconds", "float", "Aircraft event time minus requested snapshot time, in seconds."),
    ("stale_event_flag", "integer", "1 when absolute event lag exceeds the configured threshold."),
    ("latitude", "float", "Reported aircraft latitude in decimal degrees."),
    ("longitude", "float", "Reported aircraft longitude in decimal degrees."),
    ("distance_miles", "float", "Haversine distance from the configured PANC reference point."),
    ("altitude_ft", "float", "Reported altitude in feet."),
    ("ground_speed_kt", "float", "Reported ground speed in knots."),
    ("vertical_speed_fpm", "float", "Reported vertical speed in feet per minute."),
    ("track_degrees", "float", "Reported track or heading in degrees."),
    ("source_type", "string", "Normalized observation source, such as ADSB, UAT, or MLAT."),
    ("squawk", "string", "Reported transponder squawk code when available."),
    ("missing_hex_flag", "integer", "1 when the source record lacks a usable hex identifier."),
    ("missing_callsign_flag", "integer", "1 when the source record lacks a usable callsign."),
    ("deduplication_status", "string", "unique or retained_from_duplicate_group."),
    ("source_filename", "string", "Original daily JSONL filename."),
    ("source_line_number", "integer", "Original one-based JSONL line number."),
    ("source_aircraft_index", "integer", "Zero-based position in the snapshot data array."),
]

data_dictionary = pd.DataFrame(
    data_dictionary_rows,
    columns=["column_name", "logical_type", "description"],
)
data_dictionary

,column_name,logical_type,description
0,event_id,string,Deterministic identifier for the retained poin...
1,event_definition_version,string,Frozen selection-definition version; always po...
2,candidate_event_flag,integer,Selection status; always 1 in the authoritativ...
3,aircraft_key,string,"Stable aircraft identity using hex, then calls..."
4,aircraft_key_source,string,Identifier field used to construct aircraft_key.
5,flight_track_id,string,Original FR24 track identifier; equal to norma...
6,hex,string,Normalized ICAO hex identifier when available.
7,callsign,string,Normalized callsign when available.
8,fr24_id,string,Normalized Flightradar24 record or track ident...
9,snapshot_time_utc,datetime UTC,Requested five-minute collection timestamp.


## 12. Save the frozen outputs and checksums

The output files are written to `data/processed/point_event_v1`. CSV timestamps use ISO-8601 UTC values. The build summary records the configuration, raw-source manifest, output counts, and file checksums.

In [13]:
output_names = config["outputs"]
event_path = OUTPUT_DIR / output_names["point_event_table"]
rejection_path = OUTPUT_DIR / output_names["rejected_records"]
summary_path = OUTPUT_DIR / output_names["build_summary"]
dictionary_path = OUTPUT_DIR / output_names["data_dictionary"]
checksums_path = OUTPUT_DIR / output_names["checksums"]

candidate_events.to_csv(event_path, index=False, date_format="%Y-%m-%dT%H:%M:%S.%f%z")
all_rejections.to_csv(rejection_path, index=False, date_format="%Y-%m-%dT%H:%M:%S.%f%z")
data_dictionary.to_csv(dictionary_path, index=False)

config_sha256 = sha256_file(CONFIG_PATH)
raw_manifest_sha256 = source_manifest_sha256(jsonl_files)

build_summary = {
    "definition": {
        "name": config["definition"]["name"],
        "version": config["definition"]["version"],
        "status": config["definition"]["status"],
    },
    "build": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "configuration_file": str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
        "configuration_sha256": config_sha256,
        "raw_source_manifest_sha256": raw_manifest_sha256,
    },
    "source": {
        "daily_files": len(jsonl_files),
        "first_file": jsonl_files[0].name,
        "last_file": jsonl_files[-1].name,
        "snapshots": len(snapshot_df),
        "raw_aircraft_observations": len(raw_df),
        "empty_snapshots": int((snapshot_df["aircraft_count"] == 0).sum()),
        "malformed_json_rows": len(malformed_df),
    },
    "selection": {
        "qualifying_before_deduplication": len(pre_dedup),
        "duplicate_rows_removed": len(duplicate_rejections),
        "candidate_events_after_deduplication": len(candidate_events),
        "unique_aircraft_keys": int(candidate_events["aircraft_key"].nunique()),
        "stale_event_count": int(candidate_events["stale_event_flag"].sum()),
        "first_event_time_utc": candidate_events["aircraft_event_time_utc"].min().isoformat(),
        "last_event_time_utc": candidate_events["aircraft_event_time_utc"].max().isoformat(),
        "rejections_by_reason": {
            str(key): int(value)
            for key, value in all_rejections["rejection_reason"].value_counts().sort_index().items()
        },
    },
    "regression_checks": regression_table.to_dict(orient="records"),
}

with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(build_summary, handle, indent=2)
    handle.write("\n")

artifact_paths = [event_path, rejection_path, summary_path, dictionary_path]
checksum_lines = [f"{sha256_file(path)}  {path.name}" for path in artifact_paths]
checksums_path.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

saved_artifacts = pd.DataFrame(
    {
        "artifact": [path.name for path in [*artifact_paths, checksums_path]],
        "path": [str(path) for path in [*artifact_paths, checksums_path]],
        "size_bytes": [path.stat().st_size for path in [*artifact_paths, checksums_path]],
    }
)
saved_artifacts

,artifact,path,size_bytes
0,candidate_point_events_v1.csv,C:\Users\slipk\OneDrive - The George Washingto...,3060319
1,point_event_v1_rejections.csv,C:\Users\slipk\OneDrive - The George Washingto...,3871452
2,point_event_v1_build_summary.json,C:\Users\slipk\OneDrive - The George Washingto...,2091
3,point_event_v1_data_dictionary.csv,C:\Users\slipk\OneDrive - The George Washingto...,2094
4,SHA256SUMS.txt,C:\Users\slipk\OneDrive - The George Washingto...,397


## 13. Final validation summary

In [14]:
final_summary = pd.DataFrame(
    {
        "metric": [
            "Source files",
            "Snapshots",
            "Raw aircraft observations",
            "Candidate events before deduplication",
            "Duplicate records removed",
            "Candidate point events retained",
            "Unique stable aircraft keys",
            "Events with stale timestamps",
            "First event time",
            "Last event time",
        ],
        "value": [
            len(jsonl_files),
            len(snapshot_df),
            len(raw_df),
            len(pre_dedup),
            len(duplicate_rejections),
            len(candidate_events),
            candidate_events["aircraft_key"].nunique(),
            int(candidate_events["stale_event_flag"].sum()),
            candidate_events["aircraft_event_time_utc"].min().isoformat(),
            candidate_events["aircraft_event_time_utc"].max().isoformat(),
        ],
    }
)

print("point_event_v1 build completed successfully.")
display(final_summary)
display(all_rejections["rejection_reason"].value_counts().rename_axis("rejection_reason").reset_index(name="count"))
display(candidate_events.head(10))

point_event_v1 build completed successfully.


,metric,value
0,Source files,365
1,Snapshots,105120
2,Raw aircraft observations,30107
3,Candidate events before deduplication,12186
4,Duplicate records removed,1
5,Candidate point events retained,12185
6,Unique stable aircraft keys,2035
7,Events with stale timestamps,24
8,First event time,2025-01-01T02:24:59+00:00
9,Last event time,2025-12-31T20:19:58+00:00


,rejection_reason,count
0,outside_radius,15640
1,ground_speed_below_minimum,1604
2,altitude_below_minimum,572
3,altitude_at_or_above_maximum,105
4,duplicate_aircraft_snapshot,1


## Next notebook

The next stage should be implemented separately as `02_add_episode_and_recurrence_features.ipynb`. It should preserve every `event_id`, attach episode metadata without collapsing point-event rows, and calculate recurrence features using only observations that occurred before each event.